# 03 · Recomendação no mapa-múndi
    Execute os notebooks 01 e 02 para publicar as observações e as previsões.
    O perfil da base é fixo. O mês seleciona somente os valores terrestres.
    O cluster aparece no mês usado para ajustar o agrupamento. As coordenadas vêm do de-para.

## Configurar caminhos

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data/tratados/spec_analogos'
RUN = ROOT / 'data/tratados/analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Escolher o mês, a base e o nicho

In [ ]:
import plotly.express as px

from feynmans_legacy.analog_study import profile_criteria, recommend

ANOMES = None  # None usa o ultimo mes observado
CORPO = "lua"
BASE = None  # None usa a primeira base da spec selecionada
NICHO = "Todos"


## Buscar os dados do mês

In [ ]:
history = pd.read_parquet(RUN / "terra.parquet")
if ANOMES is None:
    ANOMES = int(history["anomes"].max())
values = history.loc[history["anomes"] == ANOMES].drop(columns="anomes")
if values.empty:
    if ANOMES <= history["anomes"].max():
        raise ValueError("Sem observacoes para o anomes selecionado.")
    forecast = pd.read_parquet(RUN / "previsoes.parquet")
    future = forecast.loc[forecast["anomes"] == ANOMES]
    if future.empty:
        raise ValueError("Sem previsoes publicadas para o anomes selecionado.")
    # pivot acusa chaves duplicadas nas previsoes, sem agregar os valores.
    values = future.pivot(  # noqa: PD010
        index="location_id", columns="variavel", values="valor_previsto"
    ).reset_index()
values = values.merge(pd.read_parquet(RUN / "estaticas.parquet"), on="location_id",
                      how="left", validate="one_to_one")


## Aplicar o perfil e ordenar os locais

In [ ]:
reference = pd.read_parquet(DATA / f"{CORPO}.parquet")
if BASE is None:
    if reference.empty:
        raise ValueError(f"Sem bases cadastradas na spec de {CORPO}.")
    BASE = reference["base_id"].iloc[0]
criteria = profile_criteria(DATA / "nichos.json", CORPO, NICHO, reference, BASE)
ranking = recommend(values, pd.read_parquet(RUN / "coordenadas.parquet"), criteria)
if ranking.empty:
    raise ValueError("Nenhum local com dados completos e coordenadas para essa consulta.")


## Adicionar os clusters do mês

In [ ]:
clusters = pd.read_parquet(RUN / "clusters.parquet").query("anomes == @ANOMES")
ranking = ranking.merge(clusters[["location_id", "cluster"]], on="location_id", how="left",
                        validate="one_to_one")
ranking["recomendado"] = ["Recomendado"] + ["Candidato"] * (len(ranking) - 1)


## Conferir os primeiros colocados

In [ ]:
print(ranking.head(10).to_string(index=False))


## Exibir os locais no mapa-múndi

In [ ]:
figure = px.scatter_geo(
    ranking, lat="latitude", lon="longitude", color="recomendado", hover_name="location_id",
    hover_data=["nota", "cluster", *criteria, *[f"alvo_{variable}" for variable in criteria]],
    projection="natural earth", title=f"{BASE} · {NICHO} · {ANOMES}"
)
figure.show()